# 0. User Parameters, Imports, Environment

## User Parameters

In [ ]:
import os

# Path to the directory containing the merlin benchmark workflow and examples
workflow_dir = '/lus/bnchlu1/neth/sst-benchmarks/merlin_benchmark/'


# Path to the directory where the output of the experiment runs will be stored
experiment_name = 'experiment1'


# Path to the directory containing the apptainer executable
apptainer_bin_path = '/lus/scratch/crickett/software/usr/bin'

# Path where apptainer should store its cache
apptainer_cache_dir = '/lus/bnchlu1/neth/.local/apptainer/'



# URL of the SST container to be used for the workflow and the destination path where it should be saved.
# If the destination path is already created, the container will not be downloaded and the url will be ignored.
sst_container_url = 'ghcr.io/hpc-ai-adv-dev/sst-perf-track-full:16.0.0'
sst_container_dest_name = 'sst-perf-track-full.sif'


print(f'workflow_dir: {workflow_dir}')

## Calculated 'Constants'

In [ ]:
# Full path to the SST input configuration file
sst_input_config = os.path.join(workflow_dir, 'merlin_benchmark.py')

# Directory containing per-simulation subdirectories
output_dir = os.path.join(workflow_dir, experiment_name + '_out')

# Full path to where the final .sif containerfile will live
sst_container_dest_path = os.path.join(workflow_dir, sst_container_dest_name)

# CSV file containing the consolidated experiment results after all simulations
# have been completed
experiment_result_file = os.path.join(workflow_dir, f'{experiment_name}_results.csv')

## Imports

In [ ]:
from workflow_utils import *
import os
import sys
sys.path.append(workflow_dir)
import workflow_args
import json
import glob
import humanfriendly
import pandas as pd
from plotnine import *

## Environment + Checks

In [ ]:
os.environ["PATH"] += os.pathsep + apptainer_bin_path
#os.environ["PATH"] += os.pathsep + e4s_cl_path
os.environ["LANG"] = "C.UTF-8"
os.environ["LC_ALL"] = "C.UTF-8"
os.environ['APPTAINER_CACHEDIR'] = apptainer_cache_dir

# 1. Download and Prepare Containers

## Download containers, create `.sif`

In [ ]:
cd(workflow_dir)

get_convert_to_sif()

# script expects there not to be an extension on the output file name
if os.path.exists(sst_container_dest_path):
    print(f"{sst_container_dest_path} already exists.")
else:
    run_cmd(f"./convert-to-sif.sh {sst_container_url} -f {sst_container_dest_path.replace('.sif', '')}")



## Prepare the e4s-cl profile

In [ ]:
run_cmd(f"e4s-cl profile edit --add-files {workflow_dir}")
run_cmd(f'e4s-cl profile edit --add-files {workflow_dir}/*.py')
run_cmd(f'e4s-cl profile edit --image {sst_container_dest_path}')

## Check that the profile injects a host MPI

If the selected e4s-cl profile has no MPI library bound, e4s-cl performs no MPI
substitution. The container's own MPI then finds no process manager and falls
back to *singleton init*, so every task silently becomes rank 0 of a 1-rank job
instead of one N-rank simulation.


In [ ]:
check_mpi_binding(sst_container_dest_path)

# 2. Build Benchmarks

In [ ]:
# Run Make within the container
# We have to create and mount this .conf file for SST to use when running
run_cmd('touch sstsimulator.conf')
run_in_container('make -j10', sst_container_dest_path, additional_apptainer_args=f'--bind sstsimulator.conf:{os.getenv("HOME")}/.sst/sstsimulator.conf')

# 3. Generate Argument Tuples

In [ ]:
run_specs = workflow_args.generate_merlin_run_specs(
    node_counts = (1,2,4),
    topologies=('dragonfly',),
    global_params={'stop_at': (200)},
    dragonfly_params={
        'dragonfly_hosts_per_router': (16),
        'dragonfly_num_groups': (8,16,32,64,128,256),
        'dragonfly_routers_per_group' : (32)
    },
    experiment_name=experiment_name,
)

# 4. Launch Runs

In [ ]:
FORCED = 1
os.makedirs(output_dir, exist_ok=True)

for run_spec in run_specs:
    run_output_dir = os.path.join(output_dir, run_spec.run_name)
    os.makedirs(run_output_dir, exist_ok=True)
    cd(run_output_dir)

    status_file_path = os.path.join(run_output_dir, 'status.txt')

    if os.path.exists(status_file_path):
        with open(status_file_path, 'r') as f:
            status = f.read().strip()
        if not FORCED and status in ['COMPLETED', 'SUBMITTED']:
            print(f'Skipping {run_spec.run_name} as it is already {status}')
            continue

    log_file = os.path.join(run_output_dir, 'run.log')
    error_file = os.path.join(run_output_dir, 'run.err')
    profiling_output_file = os.path.join(run_output_dir, 'profiling.json')

    param_file = os.path.join(run_output_dir, 'params.json')
    with open(param_file, 'w') as f:
        json.dump(run_spec.to_dict(), f, indent=2)


    srun_part = " ".join(run_spec.launcher["srun"])
    sst_part = " ".join(run_spec.sst_args)
    config_part = " ".join(run_spec.config_args)

    launch_cmd = (
        f"e4s-cl launch srun --output={log_file} --error={error_file} "
        f"{srun_part} \\\n\t -- sst --profiling-output={profiling_output_file} --timing-info=3 {sst_input_config} "
        f"{sst_part} \\\n\t -- {config_part}"
    )

    
    sbatch_script_path = os.path.join(run_output_dir, 'run.sbatch')

    with open(sbatch_script_path, 'w') as f:
        f.write(f"#!/bin/bash\n")
        f.write(f"echo 'LAUNCHED' > {status_file_path}\n")
        f.write(f"{launch_cmd}\n")
        f.write(f"if [ $? -eq 0 ]; then\n")
        f.write(f"    echo 'COMPLETED' > {status_file_path}\n")
        f.write(f"else\n")
        f.write(f"    echo 'FAILED' > {status_file_path}\n")
        f.write(f"fi\n")

    os.chmod(sbatch_script_path, 0o755)

    sbatch_log = os.path.join(run_output_dir, 'sbatch.log')
    sbatch_err = os.path.join(run_output_dir, 'sbatch.err')
    sbatch_cmd = f"sbatch --job-name={experiment_name} --output={sbatch_log} --error={sbatch_err} {srun_part} {sbatch_script_path}"
    sbatch_submit_file = os.path.join(run_output_dir, 'sbatch_submit_cmd.txt')
    with open(sbatch_submit_file, 'w') as f:
        f.write(f"{sbatch_cmd}\n")

    with open(status_file_path, 'w') as f:
        f.write('SUBMITTED\n')
    run_cmd(sbatch_cmd)

    cd(workflow_dir)

# 5. Wait for Job Completion

In [ ]:
wait_for_jobs(job_name=experiment_name)

# 6. Collect Simulation Results

In [ ]:
result_files = glob.glob(f'{output_dir}/**/profiling.json', recursive=True)
    
print(result_files)
columns = []
for result_file in result_files:
    column_data = {}
    param_file = result_file.replace('profiling.json', 'params.json')
    if os.path.exists(param_file):
        with open(param_file, 'r') as f:
            params = json.load(f)
            for key, value in params.items():
                column_data[key] = value

    with open(result_file, 'r') as f:
        content = json.load(f)
        regions = content['regions']
        column_data['total_duration_s'] = humanfriendly.parse_timespan(regions['total']['duration'])    
        column_data['build_duration_s'] = humanfriendly.parse_timespan(regions['total']['build']['duration'])
        column_data['execute_duration_s'] = humanfriendly.parse_timespan(regions['total']['execute']['duration'])

        column_data['total_memory_gib'] = humanfriendly.parse_size(regions['total']['total_memory']) / (1024.0 ** 3)
        column_data['build_memory_gib'] = humanfriendly.parse_size(regions['total']['build']['total_memory']) / (1024.0 ** 3)
        column_data['execute_memory_gib'] = humanfriendly.parse_size(regions['total']['execute']['total_memory']) / (1024.0 ** 3)

        column_data['global_max_rss_gib'] = humanfriendly.parse_size(content['resources']['global_max_rss']) / (1024.0 ** 3)
        column_data['local_max_rss_gib'] = humanfriendly.parse_size(content['resources']['local_max_rss']) / (1024.0 ** 3)

    param_file = result_file.replace('profiling.json', 'params.json')
    if os.path.exists(param_file):
        with open(param_file, 'r') as f:
            params = json.load(f)
            for key, value in params.items():
                column_data[key] = value
    #print(column_data)
    columns.append(column_data)


df = pd.DataFrame(columns)

df.to_csv(experiment_result_file, index=False)


# 7. Analyze Results

In [ ]:
df = pd.read_csv(experiment_result_file)
df.head()

In [ ]:
p = ggplot(df, aes(x='dragonfly_num_groups', y='build_duration_s', color='factor(node_count)'))
#p += facet_grid('dragonfly_hosts_per_router', 'dragonfly_num_groups', labeller='label_both')
p += geom_point()
p

In [ ]:

p = ggplot(df, aes(x='node_count'))
p += facet_grid('dragonfly_hosts_per_router', 'dragonfly_num_groups', labeller='label_both')
p += geom_point(aes(y='total_duration_s'), color='red')
p += geom_point(aes(y='build_duration_s'), color='blue')
p += geom_point(aes(y='execute_duration_s'), color='green')
p